# 计算市场平均方差、偏度、峰度

In [1]:
## 导入NumPy、Pandas、Wind
import numpy as np
import pandas as pd
from WindPy import w
import datetime
from dateutil.relativedelta import relativedelta

In [2]:
## 读入个股收益率各月的方差、偏度、峰度
prdct = pd.read_csv('Raw Transaction Data/prdct_dt.csv')

In [3]:
prdct.head()

,code,year,month,num,var,adj_var,math_skew,math_kurt
0,000001.SZ,2007,1,20,0.046588,0.002329,-0.897868,-0.690205
1,000001.SZ,2007,2,15,0.027840,0.001856,0.168187,-1.414870
2,000001.SZ,2007,3,22,0.009971,0.000453,0.366919,-0.376840
3,000001.SZ,2007,4,21,0.025306,0.001205,-0.734114,-0.084518
4,000001.SZ,2007,5,18,0.003345,0.000186,0.307989,-0.024779


## 等权重加权法

In [4]:
## 直接按月分组，计算数学平均值
## mean value_equivalent weighted 
mnv_ew = prdct.groupby([prdct['year'], prdct['month']])['var', 'adj_var', 'math_skew', 'math_kurt'].mean()
mnv_ew = mnv_ew.reset_index()

In [5]:
mnv_ew.to_csv('Weight and Ave/mnv_ew.csv', index=False)

In [6]:
mnv_ew

,year,month,var,adj_var,math_skew,math_kurt
0,2007,1,0.044231,0.002271,-0.143571,-0.136528
1,2007,2,0.014582,0.001003,-0.535110,0.761365
2,2007,3,0.037484,0.001738,0.122737,0.219057
3,2007,4,0.136230,0.006537,-0.232924,0.676538
4,2007,5,0.056438,0.003171,-0.358321,0.198198
...,...,...,...,...,...,...
139,2018,8,0.016769,0.000731,0.320492,0.441159
140,2018,9,0.008666,0.000460,0.042578,0.391388
141,2018,10,0.024966,0.001391,-0.455963,0.712175
142,2018,11,0.019460,0.000894,-0.061371,0.706875


## 市值加权法

### 计算各公司加权权重

In [9]:
## 直接读取样本股票代码
codes = pd.read_csv('Raw Transaction Data/AllCorps.csv', skiprows= 1, header= None)
codes = codes[0].tolist()
codes.sort()

In [11]:
## 启动Wind终端的API
## 判断WindPy是否已经登录成功
w.start() 
w.isconnected()

Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2020 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


True

In [12]:
## 读取市值数据
evdt = w.wsd(codes, "ev", "2006-12-01", "2018-11-30", "unit=1;Period=M")
ev = pd.DataFrame(evdt.Data, index = evdt.Codes, columns = evdt.Times)
ev = ev.T
ev = ev.reset_index()

In [13]:
## 转为一维表，准备合并
ev_T = pd.melt(ev, 
            id_vars = 'index', 
            value_vars = list(ev.columns[1:]), 
            var_name='code', 
            value_name='ev_month')
ev_T.columns = ['date', 'code', 'ev_month']
ev_T['date'] = pd.to_datetime(ev_T['date'])
ev_T['date'] = ev_T['date'].apply(lambda x: x + relativedelta(months = +1)) 

In [14]:
## 生成年、月
ev_T['year'] = ev_T['date'].dt.year
ev_T['month'] = ev_T['date'].dt.month

In [15]:
ev_T

,date,code,ev_month,year,month
0,2007-01-29,000001.SZ,2.815605e+10,2007,1
1,2007-02-28,000001.SZ,3.722358e+10,2007,2
2,2007-03-28,000001.SZ,3.706791e+10,2007,3
3,2007-04-30,000001.SZ,3.673712e+10,2007,4
4,2007-05-30,000001.SZ,5.049408e+10,2007,5
...,...,...,...,...,...
513643,2018-08-31,603999.SH,3.398400e+09,2018,8
513644,2018-09-30,603999.SH,3.231360e+09,2018,9
513645,2018-10-28,603999.SH,2.943360e+09,2018,10
513646,2018-11-30,603999.SH,2.776320e+09,2018,11


### 加权

In [16]:
## 合并prdct和ev_w
vw_fore = pd.merge(prdct, ev_T, on = ['code', 'year', 'month'], how = 'left')

In [17]:
## 按月分组，计算各月总市值
## 计算样本数据的权重
tt_ev = vw_fore.groupby([vw_fore['year'], vw_fore['month']])['ev_month'].agg([('TEV', 'sum'),])
tt_ev = tt_ev.reset_index()
ev_w = pd.merge(vw_fore, tt_ev, on = ['year', 'month'], how = 'left')
ev_w['weight'] = ev_w['ev_month']/ ev_w['TEV']

In [19]:
## 计算各年权重
weight = ev_T.groupby([ev_T['year'], ev_T['month']])['ev_month'].agg([('TEV', 'sum'),])
weight = weight.reset_index()
weight = pd.merge(ev_T, weight, on = ['year', 'month'], how = 'left')
weight['weight'] = weight['ev_month']/ weight['TEV']
weight = weight[['code', 'year', 'month', 'weight']]
weight[['code', 'year', 'month', 'weight']].to_csv('Weight and Ave/weight.csv', index=False)

In [20]:
ev_w

,code,year,month,num,var,adj_var,math_skew,math_kurt,date,ev_month,TEV,weight
0,000001.SZ,2007,1,20,0.046588,0.002329,-0.897868,-0.690205,2007-01-29,2.815605e+10,9.777666e+12,0.002880
1,000001.SZ,2007,2,15,0.027840,0.001856,0.168187,-1.414870,2007-02-28,3.722358e+10,1.144924e+13,0.003251
2,000001.SZ,2007,3,22,0.009971,0.000453,0.366919,-0.376840,2007-03-28,3.706791e+10,1.207539e+13,0.003070
3,000001.SZ,2007,4,21,0.025306,0.001205,-0.734114,-0.084518,2007-04-30,3.673712e+10,1.369145e+13,0.002683
4,000001.SZ,2007,5,18,0.003345,0.000186,0.307989,-0.024779,2007-05-30,5.049408e+10,1.706429e+13,0.002959
...,...,...,...,...,...,...,...,...,...,...,...,...
337178,603999.SH,2018,8,23,0.011140,0.000484,0.517311,-0.263385,2018-08-31,3.398400e+09,5.517006e+13,0.000062
337179,603999.SH,2018,9,19,0.002344,0.000123,-0.664188,-0.413938,2018-09-30,3.231360e+09,5.210388e+13,0.000062
337180,603999.SH,2018,10,18,0.073409,0.004078,0.159128,0.074828,2018-10-28,2.943360e+09,5.328611e+13,0.000055
337181,603999.SH,2018,11,22,0.015331,0.000697,-0.131413,-0.705213,2018-11-30,2.776320e+09,4.894962e+13,0.000057


In [21]:
## 筛选需要的指标列
vw = ev_w[['code', 'year', 'month', 'weight', 'var', 'adj_var', 'math_skew', 'math_kurt']]

In [22]:
## 计算加权指标
vw['var_w'] = vw['var'] * vw['weight']
vw['adjvar_w'] = vw['adj_var'] * vw['weight']
vw['mathskew_w'] = vw['math_skew'] * vw['weight']
vw['mathkurt_w'] = vw['math_kurt'] * vw['weight']

In [23]:
## 按月加总
mnv_vw = vw.groupby([vw['year'], vw['month']])['var_w', 'adjvar_w', 'mathskew_w', 'mathkurt_w'].sum()
mnv_vw = mnv_vw.reset_index()

In [24]:
mnv_vw

,year,month,var_w,adjvar_w,mathskew_w,mathkurt_w
0,2007,1,0.038083,0.001904,-0.230598,-0.060531
1,2007,2,0.014194,0.000946,-0.499410,0.358401
2,2007,3,0.017315,0.000787,0.261622,0.065908
3,2007,4,0.027041,0.001288,-0.218136,0.401851
4,2007,5,0.026346,0.001464,-0.270461,0.823381
...,...,...,...,...,...,...
139,2018,8,0.013004,0.000565,0.354465,0.348423
140,2018,9,0.006360,0.000335,0.344977,0.774344
141,2018,10,0.017640,0.000980,-0.326911,0.220151
142,2018,11,0.009962,0.000453,0.003506,0.632958


In [26]:
mnv_vw.to_csv('Weight and Ave/mnv_vw.csv', index=False)